# 00 — Getting Started with `midas_calibrate`

`midas_calibrate` is the production native-Python/Torch reference
engine for MIDAS detector-geometry calibration.  It replaces the C
chain `AutoCalibrateZarr -> CalibrantIntegratorOMP -> CalibrationCore`
with the same paramstest input format and byte-compatible output,
running on CPU.

This notebook is **fully self-contained** — it renders a synthetic
CeO2 calibrant image, runs the full E↔M alternating engine, inspects
convergence, and writes a refined paramstest.  No data files, no
network.  Runtime ~30-45 s on a CPU.

The engine:

* **E-step** — integrates the image into a 2D (R, η) cake at the
  current geometry, fits each ring's radial peak.
* **M-step** — Levenberg-Marquardt minimisation of the per-(ring, η)
  pseudo-strain residual w.r.t. geometry (Lsd, BC, tilts, distortion).
* Repeat for `nIterations`.


In [1]:
import os
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')   # macOS OpenMP guard
import numpy as np

from midas_integrate.geometry import build_tilt_matrix, pixel_to_REta
from midas_calibrate import CalibrationParams, build_ring_table


def make_truth() -> CalibrationParams:
    """Known-truth CeO2 geometry on a small 1024x1024 detector."""
    p = CalibrationParams()
    p.NrPixelsY = 1024; p.NrPixelsZ = 1024
    p.pxY = 200.0; p.pxZ = 200.0
    p.Lsd = 1_000_000.0
    p.BC_y = 512.0; p.BC_z = 512.0
    p.tx = 0.0; p.ty = 0.4; p.tz = 0.25
    p.Wavelength = 0.173
    p.SpaceGroup = 225
    p.LatticeConstant = (5.411, 5.411, 5.411, 90.0, 90.0, 90.0)
    p.MaxRingRad = 480.0
    p.MinRingRad = 0.0
    p.RhoD = 512.0
    p.Width = 1500.0
    p.EtaBinSize = 10.0
    p.RBinSize = 1.0
    p.nIterations = 4
    p.RemoveOutliersBetweenIters = False
    p.SNRMin = 1.5
    p.tolLsd = 5000.0; p.tolBC = 8.0; p.tolTilts = 1.0
    p.tolDistortion = 0.0
    p.Refine = {
        'Lsd': True, 'BC': True, 'ty': True, 'tz': True,
        'Wavelength': False, 'Parallax': False,
        **{f'p{i}': False for i in range(15)},
    }
    return p


def simulate_image(params: CalibrationParams, ring_thickness_px: float = 1.5) -> np.ndarray:
    """Render a 2D image: bright Gaussian rings on a noisy background."""
    rt = build_ring_table(params)
    NY, NZ = params.NrPixelsY, params.NrPixelsZ
    px = 0.5 * (params.pxY + params.pxZ)
    TRs = build_tilt_matrix(params.tx, params.ty, params.tz)
    Y_grid, Z_grid = np.meshgrid(np.arange(NY, dtype=np.float64),
                                 np.arange(NZ, dtype=np.float64))
    R_pix, _ = pixel_to_REta(
        Y_grid, Z_grid, Ycen=params.BC_y, Zcen=params.BC_z, TRs=TRs,
        Lsd=params.Lsd, RhoD=params.RhoD, px=px, parallax=params.Parallax,
    )
    img = np.full(R_pix.shape, 50.0, dtype=np.float64)
    rng = np.random.default_rng(0)
    img += rng.normal(0, 5.0, size=img.shape)
    for r_ideal in rt.r_ideal_px:
        I_amp = 1000.0 / (1.0 + r_ideal / 100.0)
        img += I_amp * np.exp(-0.5 * ((R_pix - r_ideal) / ring_thickness_px) ** 2)
    return img


## Step 1 — Render the synthetic calibrant image

`make_truth()` defines a known geometry; `simulate_image()` projects
the CeO2 ring table onto a tilted 1024x1024 detector.


In [2]:
truth = make_truth()
image = simulate_image(truth)
print(f'image: shape={image.shape}, min={image.min():.0f}, '
      f'max={image.max():.0f}, mean={image.mean():.0f}')

rt = build_ring_table(truth)
print(f'CeO2 rings in field of view: {len(rt.r_ideal_px)}')
print(f'truth: Lsd={truth.Lsd:.0f} um  BC=({truth.BC_y},{truth.BC_z})  '
      f'ty={truth.ty}  tz={truth.tz}')


image: shape=(1024, 1024), min=27, max=332, mean=58


CeO2 rings in field of view: 13
truth: Lsd=1000000 um  BC=(512.0,512.0)  ty=0.4  tz=0.25


## Step 2 — Build a perturbed seed

A real calibration starts from an approximate geometry.  We perturb
the truth: +300 um on Lsd, ~1.5 px on BC, ~0.05 deg on the tilts.
This is the seed handed to `autocalibrate`.


In [3]:
seed = make_truth()
seed.Lsd += 300.0
seed.BC_y += 1.5
seed.BC_z -= 1.0
seed.ty -= 0.05
seed.tz += 0.06
print(f'seed:  Lsd={seed.Lsd:.0f}  BC=({seed.BC_y},{seed.BC_z})  '
      f'ty={seed.ty:.3f}  tz={seed.tz:.3f}')


seed:  Lsd=1000300  BC=(513.5,511.0)  ty=0.350  tz=0.310


## Step 3 — Run the full E↔M engine

`autocalibrate(params, image)` returns a `CalibrationResult` with the
refined `params` and a per-iteration `history`.


In [4]:
import time
from midas_calibrate import autocalibrate

t0 = time.time()
result = autocalibrate(seed, image, verbose=False)
elapsed = time.time() - t0

final = result.history[-1]
print(f'elapsed: {elapsed:.1f} s   ({len(result.history)} iterations)')
print(f'final mean strain: {final.mean_strain_uE:.1f} ue')
print(f'fits used in last iter: {final.n_fitted}')


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


/Users/hsharma/opt/MIDAS/packages/midas_integrate/midas_integrate/kernels.py:396: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/Context.cpp:767.)
  return torch.sparse_csr_tensor(indptr, indices, values,
/Users/hsharma/opt/MIDAS/packages/midas_integrate/midas_integrate/kernels.py:396: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:51.)
  return torch.sparse_csr_tensor(indptr, in

elapsed: 38.5 s   (4 iterations)
final mean strain: 21.6 ue
fits used in last iter: 176


## Step 4 — Inspect convergence

The `history` records strain and geometry at every iteration.  A
healthy calibration shows the pseudo-strain dropping monotonically and
the geometry locking onto the truth.


In [5]:
print(f'{"iter":>4s}  {"strain (ue)":>12s}  {"Lsd (um)":>12s}  '
      f'{"BC_y":>9s}  {"BC_z":>9s}  {"ty":>8s}  {"tz":>8s}  {"n_fit":>6s}')
for h in result.history:
    print(f'{h.iteration:>4d}  {h.mean_strain_uE:>12.1f}  {h.Lsd:>12.1f}  '
          f'{h.BC_y:>9.4f}  {h.BC_z:>9.4f}  {h.ty:>8.4f}  {h.tz:>8.4f}  '
          f'{h.n_fitted:>6d}')


iter   strain (ue)      Lsd (um)       BC_y       BC_z        ty        tz   n_fit
   0         105.2     1000219.4   512.1977   511.9051    0.3433    0.1802     176
   1          25.7      999973.5   512.0047   511.9988    0.4029    0.2505     176
   2          19.4      999946.1   511.9952   512.0004    0.4004    0.2672     176
   3          21.6      999918.1   511.9906   512.0038    0.3921    0.2850     176


In [6]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

iters = [h.iteration for h in result.history]
strain = [h.mean_strain_uE for h in result.history]
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(iters, strain, 'o-', color='C0')
ax.set_xlabel('iteration')
ax.set_ylabel('mean pseudo-strain (ue)')
ax.set_yscale('log')
ax.set_title('midas_calibrate convergence (synthetic CeO2)')
ax.grid(alpha=0.3)
fig.tight_layout()
out_png = 'getting_started_convergence.png'
fig.savefig(out_png, dpi=120)
plt.close(fig)
print(f'wrote {out_png}')


wrote getting_started_convergence.png


## Step 5 — Recovery vs truth

Compare the refined geometry to the known truth.  Lsd and BC should
recover to sub-pixel / sub-100-um accuracy; the tilts to a few
millidegrees.


In [7]:
p = result.params
print(f'{"param":<8s}  {"truth":>12s}  {"refined":>12s}  {"error":>12s}')
print(f'{"Lsd":<8s}  {truth.Lsd:>12.2f}  {p.Lsd:>12.2f}  {p.Lsd-truth.Lsd:>+12.2f} um')
print(f'{"BC_y":<8s}  {truth.BC_y:>12.4f}  {p.BC_y:>12.4f}  {p.BC_y-truth.BC_y:>+12.4f} px')
print(f'{"BC_z":<8s}  {truth.BC_z:>12.4f}  {p.BC_z:>12.4f}  {p.BC_z-truth.BC_z:>+12.4f} px')
print(f'{"ty":<8s}  {truth.ty:>12.4f}  {p.ty:>12.4f}  {p.ty-truth.ty:>+12.4f} deg')
print(f'{"tz":<8s}  {truth.tz:>12.4f}  {p.tz:>12.4f}  {p.tz-truth.tz:>+12.4f} deg')


param            truth       refined         error
Lsd         1000000.00     999918.07        -81.93 um
BC_y          512.0000      511.9906       -0.0094 px
BC_z          512.0000      512.0038       +0.0038 px
ty              0.4000        0.3921       -0.0079 deg
tz              0.2500        0.2850       +0.0350 deg


## Step 6 — Write the refined paramstest

`result.params.write(path)` writes a paramstest in the same format the
C MIDAS pipeline reads — directly consumable by downstream integration
and FF/NF-HEDM analysis.


In [8]:
import tempfile, os
out_dir = tempfile.mkdtemp(prefix='midas_calib_nb_')
out_path = os.path.join(out_dir, 'calib_refined.txt')
result.params.write(out_path)
print(f'wrote refined paramstest: {out_path}')
print('--- first lines ---')
with open(out_path) as f:
    for line in list(f)[:12]:
        print(line.rstrip())


wrote refined paramstest: /var/folders/qw/k6gzh2ws7w397493kq4vnl_w0001pb/T/midas_calib_nb_y6f4sql3/calib_refined.txt
--- first lines ---
Lsd 999918.066615
BC 511.990567 512.003828
tx 0.000000
ty 0.392131
tz 0.284979
p0 1.595236859e-10
p1 -8.938314114e-15
p2 -2e-09
p3 2.138748865e-09
p4 3.248632328e-18
p5 -2.488936271e-13
p6 3.5472183e-11


## What you learned

1. `autocalibrate(params, image)` runs the full alternating E↔M engine.
2. `result.history` carries per-iteration strain + geometry for
   convergence diagnostics.
3. `result.params.write(path)` emits a C-compatible refined paramstest.

See **01_v1_vs_v2_comparison** for a head-to-head against the
differentiable `midas_calibrate_v2` engine on the same synthetic image.
